In [37]:
# !pip install .

In [39]:
import numpy as np
import pandas as pd
from statsmodels.robust import mad
import os
import time

from sklearn.linear_model import LassoLarsIC, Lasso, LassoCV

from epftoolbox.data import scaling
from epftoolbox.data import read_data
from epftoolbox.evaluation import MAE, sMAPE

from sklearn.utils._testing import ignore_warnings
from sklearn.exceptions import ConvergenceWarning

In [40]:
class LEAR(object):
    """Class to build a LEAR model, recalibrate it, and use it to predict DA electricity prices.
    
    An example on how to use this class is provided :ref:`here<learex2>`.
    
    Parameters
    ----------
    calibration_window : int, optional
        Calibration window (in days) for the LEAR model.
        
    """
    
    def __init__(self, calibration_window=364 * 3):

        # Calibration window in hours
        self.calibration_window = calibration_window
        self._const_idx   = None   
        self._center_idx  = None   
        self._scale_idx   = None 
        #self._dup_idx = None

    # Ignore convergence warnings from scikit-learn LASSO module
    @ignore_warnings(category=ConvergenceWarning)
    def recalibrate(self, Xtrain, Ytrain):
        """Function to recalibrate the LEAR model. 
        
        It uses a training (Xtrain, Ytrain) pair for recalibration
        
        Parameters
        ----------
        Xtrain : numpy.array
            Input in training dataset. It should be of size *[n,m]* where *n* is the number of days
            in the training dataset and *m* the number of input features
        
        Ytrain : numpy.array
            Output in training dataset. It should be of size *[n,24]* where *n* is the number of days 
            in the training dataset and 24 are the 24 prices of each day
                
        Returns
        -------
        numpy.array
            The prediction of day-ahead prices after recalibrating the model        
        
        """

        # # Applying Invariant, aka asinh-median transformation to the prices
        [Ytrain], self.scalerY = scaling([Ytrain], 'Invariant')
        #self.scalerY = True ## for congestion
        
        # # Rescaling all inputs except dummies (7 last features)
        # [Xtrain_no_dummies], self.scalerX = scaling([Xtrain[:, :-7]], 'Invariant')
        # Xtrain[:, :-7] = Xtrain_no_dummies
        
        #self.scalerX = None

        ## delete constant column if have
        
        non_dummy = np.arange(Xtrain.shape[1] - 7)             # last 7 weekday dummies
        const_mask = np.ptp(Xtrain[:, non_dummy], axis=0) == 0
        self._const_idx = non_dummy[const_mask]                
        
        if self._const_idx.size:
            Xtrain = np.delete(Xtrain, self._const_idx, axis=1)
        
        # 
        non_dummy_new = np.arange(Xtrain.shape[1] - 7)
        centered = Xtrain - Xtrain.mean(axis=0, keepdims=True)
        center_mask = np.ptp(centered[:, non_dummy_new], axis=0) == 0
        self._center_idx = non_dummy_new[center_mask]          
        
        if self._center_idx.size:
            Xtrain = np.delete(Xtrain, self._center_idx, axis=1)
            
        # non_dummy_dup = np.arange(Xtrain.shape[1] - 7)
        # df_dup = pd.DataFrame(Xtrain[:, non_dummy_dup])
        # dup_mask = df_dup.T.duplicated().to_numpy()
        # self._dup_idx = non_dummy_dup[dup_mask]
        # if self._dup_idx.size:
        #     Xtrain = np.delete(Xtrain, self._dup_idx, axis=1)
        # else:
        #     self._dup_idx = None
        #  only scale for mad not equal to 0
        non_dummy_final = np.arange(Xtrain.shape[1] - 7)
        safe_cols = [i for i in non_dummy_final if mad(Xtrain[:, i]) != 0]
        self._scale_idx = np.array(safe_cols, dtype=int)

        print("Deleted constant columns (index):", self._const_idx)
        print("Deleted centered-constant columns (index):", self._center_idx)
        
        if self._scale_idx.size:
            [X_scaled], self.scalerX = scaling([Xtrain[:, self._scale_idx]], 'Invariant')
            Xtrain[:, self._scale_idx] = np.nan_to_num(X_scaled, nan=0.0)
        else:
            self.scalerX = None



        
        # print("\n Checking MAD == 0 columns in Xtrain:")
        # mad_vals = self.scalerX.scaler.mad
        # zero_mad_cols = np.where(mad_vals == 0)[0]
        # print(" MAD = 0 at columns:", zero_mad_cols)

        # self.models = {}
        # for h in range(24):

        #     # Estimating lambda hyperparameter using LARS
        #     param_model = LassoLarsIC(criterion='aic', max_iter=2500)
        #     param = param_model.fit(Xtrain, Ytrain[:, h]).alpha_

        #     # Re-calibrating LEAR using standard LASSO estimation technique
        #     model = Lasso(max_iter=2500, alpha=param)
        #     model.fit(Xtrain, Ytrain[:, h])

        #     self.models[h] = model

        alpha_24 = np.zeros(24)
        # coef_24 = np.zeros((24, 370))
        coef_24 = np.zeros((24, 247))
        
        self.models = {}
        for h in range(24):
            y_h = Ytrain[:, h]
            
            try:
                # ===== 原做法 =====
                param_model = LassoLarsIC(criterion='aic', max_iter=2500)
                alpha = param_model.fit(Xtrain, y_h).alpha_
                model = Lasso(max_iter=2500, alpha=alpha).fit(Xtrain, y_h)
                alpha_24[h] = alpha
                coef_24[h, :] = model.coef_
                
            except ValueError as e:

                if "operands could not be broadcast" in str(e):
                    print(f"[WARN] LARS failed on hour {h}. Falling back to LassoCV.")
                    model = LassoCV(cv=5, max_iter=2500, n_jobs=-1).fit(Xtrain, y_h)
                else:
                    raise

            self.models[h] = model

        return alpha_24, coef_24 
        
    def predict(self, X):
        """Function that makes a prediction using some given inputs.
        
        Parameters
        ----------
        X : numpy.array
            Input of the model.
        
        Returns
        -------
        numpy.array
            An array containing the predictions.
        """

        # Predefining predicted prices
        Yp = np.zeros(24)

        # # Rescaling all inputs except dummies (7 last features)
        # X_no_dummies = self.scalerX.transform(X[:, :-7])
        # X[:, :-7] = X_no_dummies

        # ---- 与训练阶段相同的列做同样缩放 ----
        # --- 先按训练阶段的两级索引删列 ---
        X_tmp = np.delete(X,   self._const_idx,  axis=1) if self._const_idx  is not None else X.copy()
        X_tmp = np.delete(X_tmp, self._center_idx, axis=1) if self._center_idx is not None else X_tmp
        #X_tmp = np.delete(X_tmp, self._dup_idx,    axis=1) if self._dup_idx    is not None else X_tmp
        # --- 再做与训练相同的 Invariant 缩放 ---
        if self.scalerX is not None and self._scale_idx is not None and self._scale_idx.size:
            X_tmp[:, self._scale_idx] = self.scalerX.transform(X_tmp[:, self._scale_idx])




        # Predicting the current date using a recalibrated LEAR
        
        for h in range(24):

            # Predicting test dataset and saving
            Yp[h] = self.models[h].predict(X_tmp)[0]
            #print(f"Hour {h}, prediction shape: {self.models[h].predict(X).shape}, value: {self.models[h].predict(X)}")
        
        Yp = self.scalerY.inverse_transform(Yp.reshape(1, -1))
        Yp = Yp.reshape(1,-1)

        return Yp

    def recalibrate_predict(self, Xtrain, Ytrain, Xtest):
        """Function that first recalibrates the LEAR model and then makes a prediction.

        The function receives the training dataset, and trains the LEAR model. Then, using
        the inputs of the test dataset, it makes a new prediction.
        
        Parameters
        ----------
        Xtrain : numpy.array
            Input of the training dataset.
        Xtest : numpy.array
            Input of the test dataset.
        Ytrain : numpy.array
            Output of the training dataset.
        
        Returns
        -------
        numpy.array
            An array containing the predictions in the test dataset.
        """

        alpha_24, coef_24 = self.recalibrate(Xtrain=Xtrain, Ytrain=Ytrain)    

        Yp = self.predict(X=Xtest)

        return alpha_24, coef_24, Yp

    def _build_and_split_XYs(self, df_train, df_test=None, date_test=None, naive_exog=False):
        
        """Internal function that generates the X,Y arrays for training and testing based on pandas dataframes
        
        Parameters
        ----------
        df_train : pandas.DataFrame
            Pandas dataframe containing the training data
        
        df_test : pandas.DataFrame
            Pandas dataframe containing the test data
        
        date_test : datetime, optional
            If given, then the test dataset is only built for that date
        
        Returns
        -------
        list
            [Xtrain, Ytrain, Xtest] as the list containing the (X,Y) input/output pairs for training, 
            and the input for testing
        """
        
        # Checking that the first index in the dataframes corresponds with the hour 00:00 
        if df_train.index[0].hour != 0 or df_test.index[0].hour != 0:
            print('Problem with the index')

        # 
        # Defining the number of Exogenous inputs
        n_exogenous_inputs = len(df_train.columns) - 1

        # 96 prices + n_exogenous * (24 * 3 exogeneous) + 7 weekday dummies
        # Price lags: D-1, D-2, D-3, D-7
        # Exogeneous inputs lags: D, D-1, D-7
        n_features = 24*4 + 7 + n_exogenous_inputs * 24*3
        # 加 4 维 Fourier（日内 2 + 周内 2）和 1 维趋势
        # n_fourier = 4
        # n_trend   = 1
        # n_features = 24*4 + 7 + n_exogenous_inputs * 24*3 + n_fourier + n_trend
    
        # Extracting the predicted dates for testing and training. We leave the first week of data
        # out of the prediction as we the maximum lag can be one week
        
        # We define the potential time indexes that have to be forecasted in training
        # and testing
        indexTrain = df_train.loc[df_train.index[0] + pd.Timedelta(weeks=1):].index ## contain 1 week lag / week =1 

        # For testing, the test dataset is different whether depending on whether a specific test
        # dataset is provided
        if date_test is None:
            indexTest = df_test.loc[df_test.index[0] + pd.Timedelta(weeks=1):].index ## no specified, just based on the date in the df_test
        ## and forecast to the end. So df_test should include up to a week ago data.
        else:
            indexTest = df_test.loc[date_test:date_test + pd.Timedelta(hours=23)].index ## specified, test for that day

        # We extract the prediction dates/days.
        predDatesTrain = indexTrain.round('1h')[::24]                
        predDatesTest = indexTest.round('1h')[::24]
        print(predDatesTest)
        # We create two dataframe to build XY.
        # These dataframes have as indices the first hour of the day (00:00)
        # and the columns represent the 23 possible horizons/dates along a day
        indexTrain = pd.DataFrame(index=predDatesTrain, columns=['h' + str(hour) for hour in range(24)])
        indexTest = pd.DataFrame(index=predDatesTest, columns=['h' + str(hour) for hour in range(24)])
        for hour in range(24):
            indexTrain.loc[:, 'h' + str(hour)] = indexTrain.index + pd.Timedelta(hours=hour)
            indexTest.loc[:, 'h' + str(hour)] = indexTest.index + pd.Timedelta(hours=hour)

        #print(indexTest)
        # Preallocating in memory the X and Y arrays          
        Xtrain = np.zeros([indexTrain.shape[0], n_features])
        Xtest = np.zeros([indexTest.shape[0], n_features])
        Ytrain = np.zeros([indexTrain.shape[0], 24])

        # --- 插入开始：平滑周期 & 趋势特征 ---
        # 训练集上按日（24h）和周（7d）做 Fourier，再加一个归一化天数趋势
        # hours_train      = indexTrain.index.hour.values
        # dows_train       = indexTrain.index.dayofweek.values
        # days_since_train = (indexTrain.index - indexTrain.index[0]).days.values
        # fourier_train = np.column_stack([
        #     np.sin(2*np.pi * hours_train / 24),
        #     np.cos(2*np.pi * hours_train / 24),
        #     np.sin(2*np.pi * dows_train  / 7),
        #     np.cos(2*np.pi * dows_train  / 7),
        # ])
        # trend_train = (days_since_train / days_since_train.max()).reshape(-1, 1)

        # # 测试集同理
        # hours_test       = indexTest.index.hour.values
        # dows_test        = indexTest.index.dayofweek.values
        # days_since_test  = (indexTest.index - indexTrain.index[0]).days.values
        # fourier_test = np.column_stack([
        #     np.sin(2*np.pi * hours_test / 24),
        #     np.cos(2*np.pi * hours_test / 24),
        #     np.sin(2*np.pi * dows_test  / 7),
        #     np.cos(2*np.pi * dows_test  / 7),
        # ])
        # trend_test = (days_since_test / days_since_train.max()).reshape(-1, 1)

        # # 把它们塞进 Xtrain/Xtest 的最后 5 列（确保 n_features 里已加 5）
        # Xtrain[:, -5:-1] = fourier_train
        # Xtrain[:,    -1] = trend_train.squeeze()

        # Xtest[:,  -5:-1] = fourier_test
        # Xtest[:,   -1] = trend_test.squeeze()


        # Index that 
        feature_index = 0
        
        #
        # Adding the historial prices during days D-1, D-2, D-3, and D-7
        #

        # For each hour of a day
        for hour in range(24):
            # For each possible past day where prices can be included
            for past_day in [1, 2, 3,7]:

                # We define the corresponding past time indexs using the auxiliary dataframses 
                pastIndexTrain = pd.to_datetime(indexTrain.loc[:, 'h' + str(hour)].values) - \
                    pd.Timedelta(hours=24 * past_day)
                pastIndexTest = pd.to_datetime(indexTest.loc[:, 'h' + str(hour)].values) - \
                    pd.Timedelta(hours=24 * past_day)

                # We include the historical prices at day D-past_day and hour "h" 
                Xtrain[:, feature_index] = df_train.loc[pastIndexTrain, 'Price']
                Xtest[:, feature_index] = df_test.loc[pastIndexTest, 'Price']
                feature_index += 1

        #
        # Adding the exogenous inputs during days D, D-1,  D-7
        #
        # For each hour of a day
        for hour in range(24):
            # For each possible past day where exogenous inputs can be included
            for past_day in [1, 7]:
                # For each of the exogenous input
                for exog in range(1, n_exogenous_inputs + 1):

                    # Definying the corresponding past time indexs using the auxiliary dataframses 
                    pastIndexTrain = pd.to_datetime(indexTrain.loc[:, 'h' + str(hour)].values) - \
                        pd.Timedelta(hours=24 * past_day)
                    pastIndexTest = pd.to_datetime(indexTest.loc[:, 'h' + str(hour)].values) - \
                        pd.Timedelta(hours=24 * past_day)

                    # Including the exogenous input at day D-past_day and hour "h" 
                    Xtrain[:, feature_index] = df_train.loc[pastIndexTrain, 'Exogenous ' + str(exog)]                    
                    Xtest[:, feature_index] = df_test.loc[pastIndexTest, 'Exogenous ' + str(exog)]
                    feature_index += 1

            # For each of the exogenous inputs we include feature if feature selection indicates it
        # for hour in range(24):
        #     for exog in range(1, n_exogenous_inputs + 1):
        #         futureIndexTrain = pd.to_datetime(indexTrain.loc[:, 'h' + str(hour)].values)
        #         # For test set, if naive_exog is True, use previous day's 23:00 value as constant
        #         if naive_exog:
        #             # Determine the constant time: for the forecast day, use the previous day at 23:00
        #             # Here date_test is the forecast day start (if provided) or derived from indexTest
        #             if date_test is not None:
        #                 const_time = (pd.to_datetime(date_test) - pd.Timedelta(days=1)).replace(hour=23, minute=0, second=0, microsecond=0)
        #             else:
        #                 # If no specific date_test provided, take the first day in indexTest and subtract one day
        #                 const_time = (pd.to_datetime(indexTest.index[0]) - pd.Timedelta(days=1)).replace(hour=23, minute=0, second=0, microsecond=0)
        #             # For Xtest, use constant value from const_time
        #             if const_time in df_test.index:
        #                 constant_value = df_test.loc[const_time, 'Exogenous ' + str(exog)]
        #             else:
        #                 constant_value = df_train.loc[const_time, 'Exogenous ' + str(exog)]
        #             Xtest[:, feature_index] = constant_value
        #         else:
        #             futureIndexTest = pd.to_datetime(indexTest.loc[:, 'h' + str(hour)].values)
        #             Xtest[:, feature_index] = df_test.loc[futureIndexTest, 'Exogenous ' + str(exog)]
        #         # For training, always use the actual value at day D
        #         Xtrain[:, feature_index] = df_train.loc[futureIndexTrain, 'Exogenous ' + str(exog)]
        #         feature_index += 1
        for hour in range(24):
            for exog in range(1, n_exogenous_inputs + 1):
                futureIndexTrain = pd.to_datetime(indexTrain.loc[:, 'h' + str(hour)].values)
                Xtrain[:, feature_index] = df_train.loc[futureIndexTrain, 'Exogenous ' + str(exog)]

    
                futureIndexTest = pd.to_datetime(indexTest.loc[:, 'h' + str(hour)].values)

                naive_feature_names = ['Exogenous 4']  # ['Exogenous 1', 'Exogenous 3']

                exog_name = 'Exogenous ' + str(exog)
                if naive_exog and exog_name in naive_feature_names:
                    Xtest[:, feature_index] = df_test.loc[
                        futureIndexTest - pd.Timedelta(hours=24),
                        exog_name
                    ]
                else:
                    Xtest[:, feature_index] = df_test.loc[
                        futureIndexTest,
                        exog_name
                    ]

                feature_index += 1

        #
        # Adding the dummy variables that depend on the day of the week. Monday is 0 and Sunday is 6
        #
        # For each day of the week
        for dayofweek in range(7):
            Xtrain[indexTrain.index.dayofweek == dayofweek, feature_index] = 1
            Xtest[indexTest.index.dayofweek == dayofweek, feature_index] = 1
            feature_index += 1

        # Extracting the predicted values Y
        for hour in range(24):
            # Defining time index at hour h
            futureIndexTrain = pd.to_datetime(indexTrain.loc[:, 'h' + str(hour)].values)
            futureIndexTest = pd.to_datetime(indexTest.loc[:, 'h' + str(hour)].values)

            # Extracting Y value based on time indexs
            Ytrain[:, hour] = df_train.loc[futureIndexTrain, 'Price']     

        # print("Naive forecast for exogenous variables (sample) in Xtest:")
        # print(Xtest[0, 576:675])

        return Xtrain, Ytrain, Xtest


    def recalibrate_and_forecast_next_day(self, df, calibration_window, next_day_date,naive_exog=False):
        """Easy-to-use interface for daily recalibration and forecasting of the LEAR model.
        
        The function receives a pandas dataframe and a date. Usually, the data should
        correspond with the date of the next-day when using for daily recalibration.
        
        Parameters
        ----------
        df : pandas.DataFrame
            Dataframe of historical data containing prices and *N* exogenous inputs. 
            The index of the dataframe should be dates with hourly frequency. The columns 
            should have the following names ``['Price', 'Exogenous 1', 'Exogenous 2', ...., 'Exogenous N']``.
        
        calibration_window : int
            Calibration window (in days) for the LEAR model.
        
        next_day_date : datetime
            Date of the day-ahead.
        
        Returns
        -------
        numpy.array
            The prediction of day-ahead prices.
        """

        # We define the new training dataset and test datasets 
        df_train = df.loc[:next_day_date - pd.Timedelta(hours=1)]
        # Limiting the training dataset to the calibration window
        df_train = df_train.iloc[-calibration_window * 24:]
    
        # We define the test dataset as the next day (they day of interest) plus the last two weeks
        # in order to be able to build the necessary input features. 
        df_test = df.loc[next_day_date - pd.Timedelta(weeks=2):, :] ## 2


        # Generating X,Y pairs for predicting prices
        Xtrain, Ytrain, Xtest, = self._build_and_split_XYs(
            df_train=df_train, df_test=df_test, date_test=next_day_date,naive_exog = naive_exog)

        # Recalibrating the LEAR model and extracting the prediction
        alpha_24, coef_24, Yp = self.recalibrate_predict(Xtrain=Xtrain, Ytrain=Ytrain, Xtest=Xtest)

        return alpha_24, coef_24, Yp

In [41]:
def evaluate_lear_in_test_dataset(path_datasets_folder=os.path.join('.', 'datasets'), 
                                  path_recalibration_folder=os.path.join('.', 'experimental_files'),
                                  dataset='PJM', years_test=2, calibration_window=364 * 3, 
                                  begin_test_date=None, end_test_date=None,naive_exog=False):
    """Function for easy evaluation of the LEAR model in a test dataset using daily recalibration. 
    
    The test dataset is defined by a market name and the test dates dates. The function
    generates the test and training datasets, and evaluates a LEAR model considering daily recalibration. 
    
    An example on how to use this function is provided :ref:`here<learex1>`.   

    Parameters
    ----------
    path_datasets_folder : str, optional
        path where the datasets are stored or, if they do not exist yet,
        the path where the datasets are to be stored.
    
    path_recalibration_folder : str, optional
        path to save the files of the experiment dataset.
    
    dataset : str, optional
        Name of the dataset/market under study. If it is one one of the standard markets, 
        i.e. ``"PJM"``, ``"NP"``, ``"BE"``, ``"FR"``, or ``"DE"``, the dataset is automatically downloaded. If the name
        is different, a dataset with a csv format should be place in the ``path_datasets_folder``.

    years_test : int, optional
        Number of years (a year is 364 days) in the test dataset. It is only used if 
        the arguments ``begin_test_date`` and ``end_test_date`` are not provided.
    
    calibration_window : int, optional
        Number of days used in the training dataset for recalibration.
    
    begin_test_date : datetime/str, optional
        Optional parameter to select the test dataset. Used in combination with the argument
        ``end_test_date``. If either of them is not provided, the test dataset is built using the 
        ``years_test`` argument. ``begin_test_date`` should either be a string with the following 
        format ``"%d/%m/%Y %H:%M"``, or a datetime object.
    
    end_test_date : datetime/str, optional
        Optional parameter to select the test dataset. Used in combination with the argument
        ``begin_test_date``. If either of them is not provided, the test dataset is built using the 
        ``years_test`` argument. ``end_test_date`` should either be a string with the following 
        format ``"%d/%m/%Y %H:%M"``, or a datetime object.       
    
    Returns
    -------
    pandas.DataFrame
        A dataframe with all the predictions in the test dataset. The dataframe is also written to path_recalibration_folder.
    """

    # Checking if provided directory for recalibration exists and if not create it
    if not os.path.exists(path_recalibration_folder):
        os.makedirs(path_recalibration_folder)

    # Defining train and testing data
    df_train, df_test = read_data(dataset=dataset, years_test=years_test, path=path_datasets_folder,
                                  begin_test_date=begin_test_date, end_test_date=end_test_date)
    #print(df_test)
    # Defining unique name to save the forecast
    forecast_file_name = 'LEAR_forecast' + '_dat' + str(dataset) + '_YT' + str(years_test) + \
                         '_CW' + str(calibration_window) + '.csv'

    forecast_file_path = os.path.join(path_recalibration_folder, forecast_file_name)


    # Defining empty forecast array and the real values to be predicted in a more friendly format
    forecast = pd.DataFrame(index=df_test.index[::24], columns=['h' + str(k) for k in range(24)])
    real_values = df_test.loc[:, ['Price']].values.reshape(-1, 24)
    print("真实值:", real_values)
    real_values = pd.DataFrame(real_values, index=forecast.index, columns=forecast.columns)
    

    forecast_dates = forecast.index

    model = LEAR(calibration_window=calibration_window)

    alpha_list = []
    coef_list = []
    
    # For loop over the recalibration dates
    for date in forecast_dates:

        start = time.time()
        
        # For simulation purposes, we assume that the available data is
        # the data up to current date where the prices of current date are not known
        data_available = pd.concat([df_train, df_test.loc[:date + pd.Timedelta(hours=23), :]], axis=0)

        # We set the real prices for current date to NaN in the dataframe of available data
        data_available.loc[date:date + pd.Timedelta(hours=23), 'Price'] = np.nan
        # print("当前预测日期：", date)
        # print("用于预测的数据日期范围：", data_available.index.min(), "至", data_available.index.max())
    

        # Recalibrating the model with the most up-to-date available data and making a prediction
        # for the next day
        
        alpha_24, coef_24, Yp = model.recalibrate_and_forecast_next_day(df=data_available, next_day_date=date, 
                                                     calibration_window=calibration_window,naive_exog = True)
        alpha_list.append(alpha_24)
        coef_list.append(coef_24)
        
        true_vals = real_values.loc[date].values
        #print("真实值:", true_vals)
        # Saving the current prediction
        forecast.loc[date, :] = Yp

        # Computing metrics up-to-current-date
        mae = np.mean(MAE(forecast.loc[:date].values.squeeze(), real_values.loc[:date].values)) 
        smape = np.mean(sMAPE(forecast.loc[:date].values.squeeze(), real_values.loc[:date].values)) * 100
        mae_current = MAE(forecast.loc[date].values.squeeze(), real_values.loc[date].values) 
        # Pringint information
        print('{} - sMAPE: {:.2f}%  |  MAE: {:.3f} | MAE_c : {:.3f}'.format(str(date)[:10], smape, mae, mae_current))

        end = time.time()
        print(f"运行时间：{end - start:.4f} 秒")
        
        # Saving forecast
        forecast.to_csv(forecast_file_path)

    return alpha_list, coef_list, forecast

In [42]:
# 假设 evaluate_lear_in_test_dataset 函数已经导入或定义在当前环境中

datasets_folder = os.path.join('.', 'datasets')
#recalib_folder = os.path.join('.', 'experimental_files') ## for first case, assume the exogenous are known for forecast horizon.
recalib_folder = os.path.join('.', 'experimental_files_naive_multivariate')

begin_test_date = '2013-06-16 00:00'
end_test_date   = '2013-12-17 23:00'


In [43]:
total_start = time.time()
alpha_list, coef_list, forecast_df = evaluate_lear_in_test_dataset(
    path_datasets_folder=datasets_folder,
    path_recalibration_folder=recalib_folder,
    dataset='gefcom_lear_data',              # 这里 "mydata" 为你的数据集名称，可根据需要修改
    #years_test=2,                  # 如果你已经指定了测试集时间，这个参数会被忽略
    calibration_window=730,    # 使用三年的数据作为校准窗口
    begin_test_date=begin_test_date,
    end_test_date=end_test_date,
    naive_exog = True #Fuel Price need too use naive forecast
)

# 输出预测结果
#print("预测结果:")
#print(forecast_df)
total_end = time.time()
print(f"运行时间：{total_end - total_start:.4f} 秒")

/Users/PeterPan/Desktop/epftoolbox-master/epftoolbox/data/_datasets.py:146: UserWarning: Parsing dates in %Y-%m-%d %H:%M format when dayfirst=True was specified. Pass `dayfirst=False` or specify a format to silence this warning.
  begin_test_date = pd.to_datetime(begin_test_date, dayfirst=True)
/Users/PeterPan/Desktop/epftoolbox-master/epftoolbox/data/_datasets.py:147: UserWarning: Parsing dates in %Y-%m-%d %H:%M format when dayfirst=True was specified. Pass `dayfirst=False` or specify a format to silence this warning.
  end_test_date = pd.to_datetime(end_test_date, dayfirst=True)


Test datasets: 2013-06-16 00:00:00 - 2013-12-17 23:00:00
真实值: [[ 34.02  28.26  24.82 ...  38.41  35.    32.31]
 [ 33.8   34.24  30.17 ...  45.27  45.07  40.  ]
 [ 41.48  37.32  33.27 ...  44.64  40.67  36.13]
 ...
 [ 74.3   70.17  63.03 ...  85.37  74.54  80.13]
 [ 69.65  61.57  71.63 ... 110.64  88.31  88.33]
 [   nan    nan    nan ...  89.02  85.4   86.13]]
DatetimeIndex(['2013-06-16'], dtype='datetime64[ns]', name='datetime', freq=None)
Deleted constant columns (index): []
Deleted centered-constant columns (index): []
2013-06-16 - sMAPE: 17.73%  |  MAE: 6.600 | MAE_c : 6.600
运行时间：9.0132 秒
DatetimeIndex(['2013-06-17'], dtype='datetime64[ns]', name='datetime', freq=None)
Deleted constant columns (index): []
Deleted centered-constant columns (index): []
2013-06-17 - sMAPE: 11.98%  |  MAE: 4.536 | MAE_c : 2.473
运行时间：9.1781 秒
DatetimeIndex(['2013-06-18'], dtype='datetime64[ns]', name='datetime', freq=None)
Deleted constant columns (index): []
Deleted centered-constant columns (index): []

In [45]:
begin_test_date = '16/06/2013 00:00'

In [46]:
start_datetime = pd.to_datetime(begin_test_date, format='%d/%m/%Y %H:%M')

In [47]:
alpha_array = np.array(alpha_list)
num_days, num_hours = alpha_array.shape
alpha_reshaped = alpha_array.reshape(num_days * num_hours)
# 生成时间序列索引（每小时一个点）
time_index = pd.date_range(start=start_datetime, periods=num_days * num_hours, freq='H')
# 生成特征列名
feature_names = ['alpha']
# 创建 DataFrame
alpha_df = pd.DataFrame(alpha_reshaped, columns=feature_names)
alpha_df['datetime'] = time_index

# 把 datetime 放最前面
cols = ['datetime'] + feature_names
alpha_df = alpha_df[cols]

/var/folders/jp/nl85t4mj1118k5q8j1b1sdkm0000gn/T/ipykernel_1457/1820626865.py:5: FutureWarning: 'H' is deprecated and will be removed in a future version, please use 'h' instead.
  time_index = pd.date_range(start=start_datetime, periods=num_days * num_hours, freq='H')


In [48]:
coef_array = np.array(coef_list)
num_days, num_hours, num_features = coef_array.shape
coef_reshaped = coef_array.reshape(num_days * num_hours, num_features)
# 生成时间序列索引（每小时一个点）
time_index = pd.date_range(start=start_datetime, periods=num_days * num_hours, freq='H')
# 生成特征列名
feature_names = [f'feature_{i}' for i in range(num_features)]
# 创建 DataFrame
coef_df = pd.DataFrame(coef_reshaped, columns=feature_names)
coef_df['datetime'] = time_index

# 把 datetime 放最前面
cols = ['datetime'] + feature_names
coef_df = coef_df[cols]

/var/folders/jp/nl85t4mj1118k5q8j1b1sdkm0000gn/T/ipykernel_1457/2708265466.py:5: FutureWarning: 'H' is deprecated and will be removed in a future version, please use 'h' instead.
  time_index = pd.date_range(start=start_datetime, periods=num_days * num_hours, freq='H')


In [49]:
forecast_df.to_csv('lear_gefcom_BANC.csv')
alpha_df.to_csv('lear_gefcom_alpha_BANC.csv')
coef_df.to_csv('lear_gefcom_coef_BANC.csv')

In [52]:
pred

,datetime,h0,h1,h2,h3,h4,h5,h6,h7,h8,...,h14,h15,h16,h17,h18,h19,h20,h21,h22,h23
0,2013-06-16,32.602796,28.765740,25.175344,23.792433,23.438528,24.481533,25.718556,26.378945,31.648351,...,48.989559,47.494196,49.028224,47.986539,46.611054,48.369156,47.187235,44.874697,40.635435,36.565121
1,2013-06-17,32.534843,28.731860,25.837308,24.726886,24.803426,27.903435,31.732668,33.303554,39.701215,...,57.029343,61.951187,61.755073,59.008581,51.971662,49.655771,46.795291,46.074519,41.238981,36.403982
2,2013-06-18,37.954101,34.063515,31.269909,29.552583,29.359923,32.401045,37.057786,39.528417,43.533377,...,63.286765,66.368942,66.152771,61.236708,54.371023,52.495995,49.494035,47.521961,43.516533,39.844847
3,2013-06-19,33.787942,30.593982,27.527370,26.036307,25.808239,27.048974,31.483214,32.925772,36.400736,...,46.412821,45.310259,46.646904,44.791289,40.433842,39.644519,39.191092,38.624971,33.580940,31.083098
4,2013-06-20,30.457926,27.183808,24.766864,24.402529,24.141394,24.533445,31.033259,34.216295,37.780313,...,50.368495,50.559039,51.905865,51.432376,47.435314,45.260403,45.069845,40.637309,34.814384,30.362430
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
180,2013-12-13,84.155397,78.578818,70.212606,68.237074,72.362914,84.252766,131.820982,164.475524,154.521488,...,114.341013,115.580410,142.751424,182.369275,162.761078,151.115318,134.209859,119.787737,104.011726,92.920586
181,2013-12-14,122.510446,111.417786,104.508460,99.672946,100.908729,96.009268,98.998794,109.437127,131.928152,...,110.353503,106.776089,132.552296,165.352036,147.651817,135.864922,133.518424,126.762124,118.593202,102.379000
182,2013-12-15,79.621528,74.778000,68.508215,64.610287,61.606723,57.301964,56.762635,58.090053,61.285071,...,62.484288,64.794558,80.440725,113.460085,95.476879,85.944992,84.000060,84.857075,73.188543,75.287648
183,2013-12-16,75.064551,71.978367,67.482934,68.022583,69.720450,75.223361,88.507775,90.924345,94.472737,...,83.820520,82.946430,102.418477,141.540224,124.766100,109.460423,98.547222,90.457976,81.586024,78.264031


In [62]:
import numpy as np
import pandas as pd
from sklearn.metrics import (
    mean_absolute_percentage_error,
    root_mean_squared_error,
    mean_absolute_error
)


pred_df = pd.read_csv("lear_gefcom_BANC.csv")

pred_df["datetime"] = pd.to_datetime(pred_df["datetime"])

hour_cols = [f"h{i}" for i in range(24)]

df_long = pred_df.melt(
    id_vars=["datetime"],
    value_vars=hour_cols,
    var_name="hour",
    value_name="value"
)

# h0 -> 0, h1 -> 1, ..., h23 -> 23
df_long["hour"] = (
    df_long["hour"]
    .str.extract(r"(\d+)")[0]
    .astype(int)
)

# Construct hourly timestamp
df_long["timestamp"] = (
    df_long["datetime"]
    + pd.to_timedelta(df_long["hour"], unit="h")
)

df_long = (
    df_long[
        ["timestamp", "datetime", "hour", "value"]
    ]
    .sort_values("timestamp")
    .reset_index(drop=True)
)

# print("Prediction data:")
# print(df_long.head())



truth = pd.read_csv("datasets/gefcom_lear_data.csv")

# Your truth file already has a column called datetime
truth["timestamp"] = pd.to_datetime(truth["datetime"])

start = pd.Timestamp("2013-06-16 00:00:00")
end   = pd.Timestamp("2013-12-18 00:00:00")

test_df = truth[
    (truth["timestamp"] >= start)
    & (truth["timestamp"] < end)
].copy()



result_df = df_long.merge(
    test_df[["timestamp", "Price"]],
    on="timestamp",
    how="inner"
)

result_df = (
    result_df
    .sort_values("timestamp")
    .reset_index(drop=True)
)

print()
print("Merged data:")
print(result_df.head())

print()
print("Prediction rows:", len(df_long))
print("Truth rows:", len(test_df))
print("Merged rows:", len(result_df))



target = result_df["Price"].to_numpy()
forecast = result_df["value"].to_numpy()

mape_val = mean_absolute_percentage_error(
    target,
    forecast
)

rmse_val = root_mean_squared_error(
    target,
    forecast
)

mae_val = mean_absolute_error(
    target,
    forecast
)

print()
print("Overall metrics")
print("----------------")
print("MAPE:", mape_val)
print("RMSE:", rmse_val)
print("MAE :", mae_val)



result_df["date"] = result_df["timestamp"].dt.date

daily_metrics = pd.DataFrame([
    {
        "date": date,

        "MAPE": mean_absolute_percentage_error(
            group["Price"],
            group["value"]
        ),

        "RMSE": root_mean_squared_error(
            group["Price"],
            group["value"]
        ),

        "MAE": mean_absolute_error(
            group["Price"],
            group["value"]
        )
    }

    for date, group in result_df.groupby("date")
])

average_metrics = daily_metrics[
    ["MAPE", "RMSE", "MAE"]
].mean()

print()
print("Average daily metrics")
print("---------------------")
print(average_metrics)


Merged data:
            timestamp   datetime  hour      value  Price
0 2013-06-16 00:00:00 2013-06-16     0  32.602796  34.02
1 2013-06-16 01:00:00 2013-06-16     1  28.765740  28.26
2 2013-06-16 02:00:00 2013-06-16     2  25.175344  24.82
3 2013-06-16 03:00:00 2013-06-16     3  23.792433  22.90
4 2013-06-16 04:00:00 2013-06-16     4  23.438528  21.59

Prediction rows: 4440
Truth rows: 4440
Merged rows: 4440

Overall metrics
----------------
MAPE: 0.08642937002566674
RMSE: 10.53529866097325
MAE : 4.704838019708239

Average daily metrics
---------------------
MAPE    0.086429
RMSE    5.832807
MAE     4.704838
dtype: float64


In [ ]:
from importlib.metadata import version
print(version("numpy"))
print(version("scipy"))

1.26.0
1.15.3


In [ ]:
result_df.to_csv('lear_gefcom.csv')